# VESTIGIUM — Reproducible Advanced Geospatial Analysis

Team Ga Tidur · NASA Space Apps Challenge 2026

This notebook uses the **current repository structure**, not the original standalone ZIP package. To avoid overclaiming, each record is a satellite thermal-anomaly pixel, not a unique fire, ignition location, or burned area. Raw MODIS and VIIRS counts remain separate.

The earlier, more extensive figure-based research walkthrough is preserved as `VESTIGIUM_ADVANCED_GEOSPATIAL_REFERENCE.ipynb` and still requires its original figures and tables.

## 1. Inputs and QA

First run the ingestion and geospatial pipelines from the repository root, as documented in `scripts/README.md`.

In [ ]:
from pathlib import Path
import json, csv, sqlite3, sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'README.md').is_file() and (p/'data').is_dir()), None)
assert ROOT is not None, 'Open the notebook from inside the VESTIGIUM repository'
OUT = ROOT/'data'/'processed'/'geospatial'
DB = ROOT/'data'/'processed'/'data-engineering'/'vestigium_hotspots.sqlite'
assert (OUT/'geospatial_quality_report.json').exists(), 'Run src/pipelines/run_advanced.py first'
qa = json.loads((OUT/'geospatial_quality_report.json').read_text(encoding='utf8'))
display(qa)

## 2. Validate equal-area aggregation

In [ ]:
grid = pd.read_csv(OUT/'monthly_equal_area_25km.csv')
assert not grid.duplicated(['sensor','year','month','cell_x','cell_y']).any()
assert int(grid.detection_count.sum()) == int(qa['accepted_type0'])
assert (grid.detection_count > 0).all()
print('Grid-month records:',len(grid))
print('Source pixel detections:', int(grid.detection_count.sum()))
print('CRS: EPSG:6933 | Cell width: 25,000 m')
display(grid.head(8))

## 3. Sensor-specific contributions

In [ ]:
by_sensor = grid.groupby('sensor',as_index=False).agg(
    grid_months=('detection_count','size'),
    pixel_detections=('detection_count','sum'),
    frp_sum_mw=('frp_sum_mw','sum'))
display(by_sensor.sort_values('sensor'))
assert int(by_sensor.pixel_detections.sum()) == qa['accepted_type0']

## 4. Multi-resolution geographic grids

In [ ]:
rows=[]
for width in (1.0,0.5,0.25,0.1):
    name='monthly_geographic_'+str(width).replace('.','p')+'deg.csv'
    p=OUT/name
    d=pd.read_csv(p,usecols=['detection_count'])
    rows.append({'grid_width_degrees':width,'aggregated_rows':len(d),
                 'pixel_detections':int(d.detection_count.sum()),
                 'conserves_source_records':int(d.detection_count.sum())==qa['accepted_type0']})
display(pd.DataFrame(rows))
assert all(row['conserves_source_records'] for row in rows)

## 5. September 2015 visualization

Point size is proportional to the count in a **25 km equal-area cell**, not the number of distinct fires.

In [ ]:
a=grid[(grid.year==2015)&(grid.month==9)].copy()
if a.empty: raise ValueError('No September 2015 grid-month rows found')
fig, ax=plt.subplots(figsize=(11,5))
sample=a.nlargest(6000,'detection_count')
ax.scatter(sample.center_longitude,sample.center_latitude,
           s=sample.detection_count.clip(1,1000)/15,alpha=.32,c='firebrick')
ax.set(xlabel='Longitude (EPSG:4326)',ylabel='Latitude (EPSG:4326)',
       title='Indonesia: satellite hot-spot detections, September 2015')
plt.show()

## 6. Sensor co-location in the same 25-km cell and UTC month

Co-location is not same-fire matching.

In [ ]:
presence=a.pivot_table(index=['cell_x','cell_y'],columns='sensor',
                        values='detection_count',aggfunc='sum',fill_value=0)
for name in ('MODIS','VIIRS_SNPP','VIIRS_NOAA20'):
    if name not in presence: presence[name]=0
display(pd.DataFrame({'sensor':list(presence.columns),
                      'active_cells':[int((presence[c]>0).sum()) for c in presence]}))
print('Cells where multiple sensors recorded detections:',
      int(((presence>0).sum(axis=1)>1).sum()))

## 7. Multi-year recurrence

This describes repeated detection in a coarse spatial cell, not the number of recurring wildfires.

In [ ]:
years=grid.groupby(['sensor','cell_x','cell_y']).agg(
    years_with_detections=('year','nunique'),
    detected_pixels=('detection_count','sum')).reset_index()
display(years.nlargest(10,'years_with_detections'))

## 8. Individual pixel inspection with the SQLite index

In [ ]:
sys.path.insert(0,str(ROOT/'src'/'pipelines'))
from example_queries import fetch_hotspots
rows=fetch_hotspots(str(DB),2015,9,108,-4,119,4,limit=8,sensor='MODIS')
display(pd.DataFrame(rows))

## 9. Scientific limitations

In [ ]:
print('PASS: equal-area and geographic grids conserve source pixel counts')
print('NOT DONE: verified province boundaries / spatial join')
print('NOT DONE: unique fire event reconstruction, ignition-cause inference')
print('NOT DONE: observation-effort normalization (clouds, overpass denominator)')
print('Harmonization is a separate stage in src/pipelines/harmonize.py')